In [ ]:
import pandas as pd
import numpy as np
import os
import json
import gzip

os.chdir('../data/beauty')
os.getcwd()

In [ ]:
file = 'inter.csv'

# df = pd.read_csv(file, names=['parent_asin', 'user_id', 'rating', 'timestamp'], header=None)  # 2018
df = pd.read_csv(file)
df = df.drop_duplicates(subset=['parent_asin', 'user_id', 'timestamp'])

# unique_users = df['user_id'].unique()
# sampled_users = pd.Series(unique_users).sample(frac=0.05, random_state=42)
# sampled_users = pd.Series(unique_users).sample(n=100000, random_state=42)
# df = df[df['user_id'].isin(sampled_users)]

df.head()

In [ ]:
duplicates = df.duplicated(subset=['user_id', 'parent_asin'], keep=False)

if duplicates.any():
    print(f"Found {duplicates.sum()} records with the same user_id and parent_asin")
    print(df[duplicates].sort_values(by=['user_id', 'parent_asin']))
else:
    print("No records with the same user_id and parent_asin")

In [ ]:
print("Initial data:")
print(f"- Number of users: {df['user_id'].nunique()}")
print(f"- Number of items: {df['parent_asin'].nunique()}")
print(f"- Number of records: {len(df)}")

In [ ]:
df = df.sort_values(['user_id', 'timestamp'])

df['basket_id'] = df.groupby(['user_id', 'timestamp']).ngroup()

df = df.drop(columns=['rating'])

df.head()

In [ ]:
meta_path = 'meta.json.gz'

def parse(path):
  g = gzip.open(path, 'rb')
  for l in g:
    yield eval(l)

def getDF(path):
  i = 0
  df = {}
  for d in parse(path):
    df[i] = d
    i += 1
  return pd.DataFrame.from_dict(df, orient='index')

meta_df = getDF(meta_path)
meta_df.head()

In [ ]:
valid_asins = meta_df[meta_df['title'].notnull()]['asin'].unique()

df = df[df['parent_asin'].isin(valid_asins)]
df.head()

In [ ]:
print("Statistics before cold-start filtering:")
print(f"- Number of users: {df['user_id'].nunique()}")
print(f"- Number of items: {df['parent_asin'].nunique()}")
print(f"- Number of baskets: {df['basket_id'].nunique()}")
print(f"- Average baskets per user: {df.groupby('user_id')['basket_id'].nunique().mean():.2f}")
print(f"- Average items per basket: {df.groupby('basket_id')['parent_asin'].count().mean():.2f}")
print(f"- Number of records: {len(df)}")

In [ ]:
min_baskets_per_u = 3
min_items_per_u = 10
min_users_per_i = 10

def filter_cold_start(df):
    while True:        
        n_users = df['user_id'].nunique()
        n_items = df['parent_asin'].nunique()
        print(f"Current number of users: {n_users}, number of items: {n_items}")

        user_basket_counts = df.groupby('user_id')['basket_id'].nunique()
        user_interact_counts = df['user_id'].value_counts()
        item_interact_counts = df['parent_asin'].value_counts()

        filtered_users = user_basket_counts[user_basket_counts >= min_baskets_per_u].index
        filtered_users2 = user_interact_counts[user_interact_counts >= min_items_per_u].index
        filtered_items = item_interact_counts[item_interact_counts >= min_users_per_i].index

        if (len(filtered_users) == n_users) and (len(filtered_users2) == n_users) and (len(filtered_items) == n_items):
            break

        df = df[df['user_id'].isin(filtered_users) & df['user_id'].isin(filtered_users2) & df['parent_asin'].isin(filtered_items)].copy()

    return df

df = filter_cold_start(df)
df['basket_id'] = df.groupby(['user_id', 'basket_id']).ngroup()

In [ ]:
print("Final statistics:")
print(f"- Number of users: {df['user_id'].nunique()}")
print(f"- Number of items: {df['parent_asin'].nunique()}")
print(f"- Number of baskets: {df['basket_id'].nunique()}")
print(f"- Average baskets per user: {df.groupby('user_id')['basket_id'].nunique().mean():.2f}")
print(f"- Average items per basket: {df.groupby('basket_id')['parent_asin'].count().mean():.2f}")
print(f"- Number of records: {len(df)}")

stats = {
    "num_users": df['user_id'].nunique(),
    "num_items": df['parent_asin'].nunique(),
    "num_baskets": df['basket_id'].nunique(),
    "avg_baskets_per_user": round(df.groupby('user_id')['basket_id'].nunique().mean(), 2),
    "avg_items_per_basket": round(df.groupby('basket_id')['parent_asin'].count().mean(), 2),
    "num_records": len(df)
}

with open('stat.json', 'w', encoding='utf-8') as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)

In [ ]:
duplicates = df.duplicated(subset=['user_id', 'parent_asin'], keep=False)

if duplicates.any():
    print(f"Found {duplicates.sum()} records with the same user_id and parent_asin")
    print(df[duplicates].sort_values(by=['user_id', 'parent_asin']))
else:
    print("No records with the same user_id and parent_asin")

In [ ]:
unique_users = df['user_id'].unique()
user_id_map = {old_id: new_id for new_id, old_id in enumerate(unique_users)}
user_mapping_df = pd.DataFrame({'original_id': user_id_map.keys(), 'mapped_id': user_id_map.values()})

unique_items = df['parent_asin'].unique()
item_id_map = {old_id: new_id for new_id, old_id in enumerate(unique_items)}
item_mapping_df = pd.DataFrame({'original_id': item_id_map.keys(), 'mapped_id': item_id_map.values()})

df['user_id'] = df['user_id'].map(user_id_map)
df['item_id'] = df['parent_asin'].map(item_id_map)
df = df.drop(columns=['parent_asin'])

user_mapping_df.to_csv('user_id_mapping.csv', index=False)
item_mapping_df.to_csv('item_id_mapping.csv', index=False)

In [ ]:
df = df.sort_values(['user_id', 'timestamp'])

basket_order_map = (
    df[['user_id', 'basket_id']]
    .drop_duplicates()
    .groupby('user_id')
    .cumcount()
)

df = df.merge(
    df[['user_id', 'basket_id']].drop_duplicates().assign(basket_ord=basket_order_map),
    on=['user_id', 'basket_id'],
    how='left'
)

# df = df.drop(columns=['timestamp'])
df = df[['user_id', 'item_id', 'basket_id', 'basket_ord', 'timestamp']]
df.head()

In [ ]:
df['tag'] = 0

max_orders = df.groupby('user_id')['basket_ord'].transform('max')
user_ids = df['user_id'].unique()

np.random.seed(42)
half_size = int(len(user_ids) * 0.5)
tag1_users = np.random.choice(user_ids, size=half_size, replace=False)

tag_map = {uid: 1 if uid in tag1_users else 2 for uid in user_ids}

df.loc[df['basket_ord'] == max_orders, 'tag'] = df.loc[df['basket_ord'] == max_orders, 'user_id'].map(tag_map)
df.head()

In [ ]:
file = 'baskets_inter.csv'

df.to_csv(file, index=False)
df = pd.read_csv(file)

df.head()